In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

os.makedirs('plots', exist_ok=True)

In [ ]:
#
# Discretisation:
#   du/dt = Du * lap(u) - u*v^2 + f*(1-u)
#   dv/dt = Dv * lap(v) + u*v^2 - (f+k)*v
#
#   Laplacian — 5-point stencil:
#   lap(u)[i,j] = (u[i+1,j]+u[i-1,j]+u[i,j+1]+u[i,j-1] - 4*u[i,j]) / dx^2
#
#   Time integration: explicit forward Euler
#
#   Boundary conditions:
#   - Neumann at left/right/bottom: ghost-cell mirror
#       i=0  -> im1=1,   i=N-1 -> ip1=N-2   (same logic for j)
#   - Dirichlet at top row (i=N-1): u=1, v=0, overwritten after every step

@njit
def step(U, V, Du, Dv, f, k, dt, dx):
    N   = U.shape[0]
    U_new = np.empty_like(U)
    V_new = np.empty_like(V)
    dx2   = dx * dx

    for i in range(N - 1):          # top row set by Dirichlet below
        for j in range(N):
            im1 = i - 1 if i > 0     else 1
            ip1 = i + 1 if i < N - 1 else N - 2
            jm1 = j - 1 if j > 0     else 1
            jp1 = j + 1 if j < N - 1 else N - 2

            lap_u = (U[ip1,j] + U[im1,j] + U[i,jp1] + U[i,jm1] - 4*U[i,j]) / dx2
            lap_v = (V[ip1,j] + V[im1,j] + V[i,jp1] + V[i,jm1] - 4*V[i,j]) / dx2

            uvv = U[i,j] * V[i,j] * V[i,j]
            U_new[i,j] = U[i,j] + dt * (Du * lap_u - uvv + f * (1 - U[i,j]))
            V_new[i,j] = V[i,j] + dt * (Dv * lap_v + uvv - (f + k) * V[i,j])

    U_new[N-1, :] = 1.0
    V_new[N-1, :] = 0.0
    return U_new, V_new


def make_initial(N, noise=0.0, seed=42):
    """
    u = 0.5 everywhere.
    v = 0.25 in the central square of 20x20 physical units (= 20/dx grid cells).
    When noise > 0, it is generated on a fixed 100x100 reference grid and
    resampled to size N — so every resolution sees the same physical perturbation.
    """
    rng = np.random.default_rng(seed)

    U = np.full((N, N), 0.5)
    V = np.zeros((N, N))

    # Central square: always 20 physical units wide regardless of dx
    mid = N // 2
    sq  = N // 10          # sq grid cells = 10 physical units -> half-width 10
    V[mid-sq : mid+sq, mid-sq : mid+sq] = 0.25

    if noise > 0:
        # Build noise on the canonical 100-cell grid, then nearest-neighbour
        # resample so coarse/fine grids get the same physical perturbation.
        ref = 100
        nu  = rng.uniform(-noise, noise, (ref, ref))
        nv  = rng.uniform(-noise, noise, (ref, ref))
        if N != ref:
            idx = (np.linspace(0, ref - 1, N) + 0.5).astype(int).clip(0, ref - 1)
            nu  = nu[np.ix_(idx, idx)]
            nv  = nv[np.ix_(idx, idx)]
        U += nu
        V += nv
        np.clip(U, 0, 1, out=U)
        np.clip(V, 0, 1, out=V)

    # Always enforce top Dirichlet BC
    U[-1, :] = 1.0
    V[-1, :] = 0.0
    return U, V


def run(N, dt, dx, steps, Du=0.16, Dv=0.08, f=0.035, k=0.060, noise=0.0, seed=42):
    """Evolve the system. Physical domain = N*dx = 100. Physical time = steps*dt."""
    U, V = make_initial(N, noise=noise, seed=seed)
    for _ in range(steps):
        U, V = step(U, V, Du, Dv, f, k, dt, dx)
    return U, V


# Warm up numba JIT
_u, _v = make_initial(4)
step(_u, _v, 0.16, 0.08, 0.035, 0.060, 1.0, 1.0)
print("Ready.")

In [ ]:
# Shows both U and V. No noise.

N, dt, dx = 100, 1.0, 1.0
steps = 8000

U_init, V_init = make_initial(N, noise=0.0)

print("Running f=0.035, k=0.060 ...")
U1, V1 = run(N, dt, dx, steps, f=0.035, k=0.060, noise=0.0)

print("Running f=0.060, k=0.062 ...")
U2, V2 = run(N, dt, dx, steps, f=0.060, k=0.062, noise=0.0)

fig, axes = plt.subplots(3, 2, figsize=(7, 10))
fig.suptitle("Gray-Scott: Initial Conditions & Two Parameter Sets", fontsize=11)

axes[0,0].imshow(U_init.T, origin='lower', cmap='RdPu', vmin=0, vmax=1)
axes[0,0].set_title("Initial U", fontsize=9)
axes[0,1].imshow(V_init.T, origin='lower', cmap='YlGn', vmin=0, vmax=0.3)
axes[0,1].set_title("Initial V", fontsize=9)

im = axes[1,0].imshow(U1.T, origin='lower', cmap='RdPu', vmin=0, vmax=1)
axes[1,0].set_title("U  (f=0.035, k=0.060) — labyrinths", fontsize=9)
plt.colorbar(im, ax=axes[1,0])
im = axes[1,1].imshow(V1.T, origin='lower', cmap='YlGn', vmin=0)
axes[1,1].set_title("V  (f=0.035, k=0.060) — labyrinths", fontsize=9)
plt.colorbar(im, ax=axes[1,1])

im = axes[2,0].imshow(U2.T, origin='lower', cmap='RdPu', vmin=0, vmax=1)
axes[2,0].set_title("U  (f=0.060, k=0.062) — worms/spots", fontsize=9)
plt.colorbar(im, ax=axes[2,0])
im = axes[2,1].imshow(V2.T, origin='lower', cmap='YlGn', vmin=0)
axes[2,1].set_title("V  (f=0.060, k=0.062) — worms/spots", fontsize=9)
plt.colorbar(im, ax=axes[2,1])

plt.tight_layout()
plt.savefig('plots/plot1_params.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved plots/plot1_params.png")

In [ ]:
# Physical domain: N * dx = 100  ->  N = int(100 / dx)
# Physical time:   steps * dt = 8000  ->  steps = int(8000 / dt)
# Stability:  r = Du * dt / dx^2 = 0.16 in all cases (well below 0.5)
#
# No noise — the only initial difference between grids is the central square,
# which is placed identically in physical space. Any pattern difference is
# purely due to discretisation accuracy. Finer grids should show sharper,
# more resolved versions of the same labyrinthine pattern.

configs = [
    dict(dx=2.0, dt=4.00, label="dx=2.0, dt=4.00  (coarse)"),
    dict(dx=1.0, dt=1.00, label="dx=1.0, dt=1.00  (default)"),
    dict(dx=0.5, dt=0.25, label="dx=0.5, dt=0.25  (fine)"),
]

results_res = []
for cfg in configs:
    dx_    = cfg['dx']
    dt_    = cfg['dt']
    N_     = int(round(100.0 / dx_))
    steps_ = int(round(8000.0 / dt_))
    r      = 0.16 * dt_ / dx_**2
    print(f"{cfg['label']}  N={N_}  steps={steps_}  r={r:.3f}")
    U_ = run(N_, dt_, dx_, steps_, noise=0.0)
    results_res.append((cfg['label'], U_[0]))

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
fig.suptitle("Effect of dx and dt  (U, no noise)\n",
             fontsize=9)

for col, (label, U_) in enumerate(results_res):
    im = axes[col].imshow(U_.T, origin='lower', cmap='RdPu', vmin=0, vmax=1)
    axes[col].set_title(label, fontsize=8)
    plt.colorbar(im, ax=axes[col], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('plots/plot2_dx_dt.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved plots/plot2_dx_dt.png")

In [ ]:
N, dt, dx = 100, 1.0, 1.0
steps = 8000

noise_levels  = [0.0, 0.10, 0.20]
results_noise = []

for noise in noise_levels:
    print(f"Running noise = {noise} ...")
    U_, V_ = run(N, dt, dx, steps, noise=noise)
    results_noise.append((noise, U_))

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
fig.suptitle("Noise effects(U, f=0.035, k=0.060)", fontsize=9)

for col, (noise, U_) in enumerate(results_noise):
    im = axes[col].imshow(U_.T, origin='lower', cmap='RdPu', vmin=0, vmax=1)
    axes[col].set_title(f"noise = {noise}", fontsize=9)
    plt.colorbar(im, ax=axes[col], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('plots/plot3_noise.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved plots/plot3_noise.png")

In [ ]:
#
# f=0.020, k=0.050 is in Pearson's time-dependent regime:
# spots continuously self-replicate and travel without settling.
# Small noise used here to seed varied spot nucleation sites.

N, dt, dx = 100, 1.0, 1.0
f_anim, k_anim  = 0.020, 0.050
frames          = 80
steps_per_frame = 150

print("Initialising ...")
U_a, V_a = make_initial(N, noise=0.02, seed=0)

print("Burn-in 3000 steps ...")
for _ in range(3000):
    U_a, V_a = step(U_a, V_a, 0.16, 0.08, f_anim, k_anim, dt, dx)

fig_a, ax_a = plt.subplots(figsize=(4, 4))
ax_a.axis('off')
im_a  = ax_a.imshow(V_a.T, origin='lower', cmap='YlGn', vmin=0, vmax=0.4, animated=True)
ttl_a = ax_a.set_title('', fontsize=8)

def update(frame):
    global U_a, V_a
    for _ in range(steps_per_frame):
        U_a, V_a = step(U_a, V_a, 0.16, 0.08, f_anim, k_anim, dt, dx)
    im_a.set_data(V_a.T)
    ttl_a.set_text(f"V  f={f_anim}, k={k_anim}   t={3000 + (frame+1)*steps_per_frame}")
    return im_a, ttl_a

ani = FuncAnimation(fig_a, update, frames=frames, interval=80, blit=True)
ani.save('plots/plot4_animation.gif', writer=PillowWriter(fps=12))
plt.close(fig_a)
print("Saved plots/plot4_animation.gif")

Image('plots/plot4_animation.gif')
